In [9]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import nibabel as nib
import torch

from atlas_fc import load_age_atlases_cropped, PADDED_SPATIAL, CROPPED_SHAPE

In [10]:
CHUNK_METADATA_CSV = (
    "/lustre/disk/home/users/mfaizan/motion_correction/cycleGANS_on_2d_images/"
    "pytorch-CycleGAN-and-pix2pix/data_preprocessing/motion_grades_chunk_5_dataset_hfiltered/"
    "chunk_metadata.csv"
)
SOURCE_ROOT = (
    "/lustre/disk/home/shared/cusacklab/foundcog/bids/derivatives/"
    "faizan_motion_correction_dataset/brain_masked_cropped_hfiltered_normalized_to_common_space"
)
OUTPUT_ROOT = (
    "/lustre/disk/home/shared/cusacklab/foundcog/bids/derivatives/"
    "faizan_motion_correction_dataset/roi_timeseries_hfiltered_videos"
)
PAD = PADDED_SPATIAL[0] - CROPPED_SHAPE[0]   # 64 - 60 = 4, split 2/2 on axis 0

In [11]:
meta = pd.read_csv(CHUNK_METADATA_CSV)
runs = (
    meta[meta["task"] == "videos"]
    [["subject_id", "session_id", "run_id", "age_group", "source_volume_path"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
print(f"{len(runs)} unique video runs")
print(runs["age_group"].value_counts())
runs.head()

431 unique video runs
age_group
2mo    330
9mo    101
Name: count, dtype: int64


,subject_id,session_id,run_id,age_group,source_volume_path
0,ICC103,2,2,2mo,/lustre/disk/home/shared/cusacklab/foundcog/bi...
1,ICC103,2,1,2mo,/lustre/disk/home/shared/cusacklab/foundcog/bi...
2,ICC103,1,1,2mo,/lustre/disk/home/shared/cusacklab/foundcog/bi...
3,ICC105,1,1,2mo,/lustre/disk/home/shared/cusacklab/foundcog/bi...
4,ICC103A,1,1,9mo,/lustre/disk/home/shared/cusacklab/foundcog/bi...


In [12]:
atlases = load_age_atlases_cropped()
for group, atlas in atlases.items():
    print(f"{group}: {atlas.n_rois} active ROIs")

2mo: 400 active ROIs
9mo: 400 active ROIs


In [13]:
def load_run_volume(path: str) -> torch.Tensor:
    """(60, 72, 56, T) on disk -> zero-padded (T, 64, 72, 56) tensor."""
    data = nib.load(path).get_fdata()
    data = np.moveaxis(data, -1, 0)                             # (T, 60, 72, 56)
    lo, hi = PAD // 2, PAD - PAD // 2
    data = np.pad(data, [(0, 0), (lo, hi), (0, 0), (0, 0)])     # (T, 64, 72, 56)
    return torch.from_numpy(data).float()


def output_path(source_volume_path: str) -> str:
    rel_dir = os.path.relpath(os.path.dirname(source_volume_path), SOURCE_ROOT)
    return os.path.join(OUTPUT_ROOT, rel_dir, "roi_timeseries.npy")


def extract_and_save(row, atlases) -> str:
    atlas = atlases[row.age_group]
    volume_seq = load_run_volume(row.source_volume_path)
    roi_ts = atlas.extract_roi_timeseries(volume_seq).numpy()   # (T, n_rois)
    out_path = output_path(row.source_volume_path)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    np.save(out_path, roi_ts)
    return out_path

In [14]:
row = runs.iloc[0]
out_path = extract_and_save(row, atlases)
print(out_path)
print("shape:", np.load(out_path).shape)

/lustre/disk/home/shared/cusacklab/foundcog/bids/derivatives/faizan_motion_correction_dataset/roi_timeseries_hfiltered_videos/_subject_id_ICC103/_referencetype_standard/_run_002_session_2_task_name_videos/roi_timeseries.npy
shape: (485, 400)


In [15]:
os.makedirs(OUTPUT_ROOT, exist_ok=True)
for group, atlas in atlases.items():
    np.save(os.path.join(OUTPUT_ROOT, f"roi_labels_{group}.npy"), np.array(atlas.active_labels))